In [ ]:
# pip install lonboard geopandas matplotlib pyarrow folium mapclassify scikit-learn

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from lonboard import viz
from shapely.geometry import Polygon
import folium
from folium import plugins
import json

DATASET_DIR = Path("../dataset")
TILFLUKTSROM_DATASET = DATASET_DIR / "tilfluktsrom.gpkg"
BEFOLKNING_DATASET = DATASET_DIR / "Befolkning_42_Agder_25832_BefolkningsstatistikkRutenett1km2025_GML.gpkg"

tilfluktsrom_gdf = gpd.read_file(TILFLUKTSROM_DATASET)
befolkning_gdf = gpd.read_file(BEFOLKNING_DATASET)

viz(befolkning_gdf.to_crs("EPSG:4326"))

# curl -X 'GET' \
#   'https://api.kartverket.no/kommuneinfo/v1/fylker/42' \
#   -H 'accept: application/json'

AGDER_URL = "https://api.kartverket.no/kommuneinfo/v1/fylker/42"
AGDER_POLYGON = requests.get(AGDER_URL).json()["avgrensningsboks"]["coordinates"][0]
agder_polygon_gdf = gpd.GeoDataFrame(
    {"name": ["Agder"]},
    geometry=[Polygon(AGDER_POLYGON)],
    crs="EPSG:4326",
)

In [ ]:
buffer_lengde = 500

shelter_gdf = tilfluktsrom_gdf.to_crs("EPSG:25832").copy()
pop_gdf = befolkning_gdf.to_crs("EPSG:25832").copy()

agder_polygon_25832 = agder_polygon_gdf.to_crs(shelter_gdf.crs)
agder_geom = agder_polygon_25832.geometry.iloc[0]

shelter_gdf = shelter_gdf[shelter_gdf.within(agder_geom)].copy()

tilfluktsrom_buffer_gdf = shelter_gdf.copy()
tilfluktsrom_buffer_gdf["geometry"] = shelter_gdf.geometry.buffer(buffer_lengde)

pop_in_buffer = gpd.sjoin(pop_gdf, tilfluktsrom_buffer_gdf, how="right", predicate="within")

In [ ]:
pop_viz_4326 = pop_in_buffer.to_crs("EPSG:4326")
shelter_viz_4326 = shelter_gdf.to_crs("EPSG:4326")

center_lat = agder_polygon_gdf.geometry.centroid.y.iloc[0]
center_lon = agder_polygon_gdf.geometry.centroid.x.iloc[0]
m = folium.Map(location=[center_lat, center_lon], zoom_start=9)

for idx, row in pop_viz_4326.iterrows():
    pop_value = row.get('popTot', 0)
    if pd.isna(pop_value):
        pop_value = 0
    
    color = '#FF6B6B'
    
    popup_text = f"<b>People in cell:</b> {int(pop_value)}"
     
    folium.GeoJson(
        data={
            "type": "Feature",
            "geometry": row.geometry.__geo_interface__,
            "properties": {},
        },
        style_function=lambda x, color=color: {
            "fillColor": color,
            "color": "black",
            "weight": 0.5,
            "fillOpacity": 0.6,
        },
        popup=folium.Popup(popup_text, max_width=250),
    ).add_to(m)

for idx, row in shelter_viz_4326.iterrows():
    shelter_name = row.get('name', f'Shelter {idx}')
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        popup=f"<b>{shelter_name}</b>",
        color='darkred',
        fill=True,
        fillColor='darkred',
        fillOpacity=0.9,
        weight=2
    ).add_to(m)

legend_html = '''
     <div style="position: fixed; 
     bottom: 50px; right: 50px; width: 220px; height: 140px; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:14px; padding: 10px">
        <p style="margin: 0;"><b>Legend</b></p>
        <p style="margin: 5px 0;"><span style="background-color: #FF6B6B; padding: 5px 10px; border-radius: 3px; color: white;">Directly on Shelter</span></p>
        <p style="margin: 5px 0;"><span style="background-color: darkred; padding: 3px 8px; border-radius: 50%; color: white; display: inline-block;">●</span> Shelter Location</p>
     </div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m